In [2]:
# Test del Framework Neural Network
import numpy as np
from nn.model import Model
from nn.layers import Dense, xavier_uniform, he_uniform
from nn.activations import ReLU, Sigmoid, Tanh, Softmax, Identity
from nn.dropout import Dropout
from nn.losses import MSE, BinaryCrossEntropy, CrossEntropy, MEE
from nn.optim import SGD
from nn.regularizers import L2, L1
from nn.data_loader import load_monk, load_cup, normalize


In [3]:
# TEST 1: XOR

# Dataset XOR
X_train = np.array([[0, 0],
                     [0, 1],
                     [1, 0],
                     [1, 1]], dtype=float)

y_train = np.array([[0],
                     [1],
                     [1],
                     [0]], dtype=float)

# Build model
model = Model(
    modules=[
        Dense(2, 4, initializer=xavier_uniform),
        Tanh(),
        Dense(4, 1, initializer=xavier_uniform),
        Sigmoid()
    ],
    loss=BinaryCrossEntropy(),
    optimizer=SGD(lr=0.5),
    regularizer=L2(lam=0.001)
)

print("\nModel Structure:")
print(model)

# Training loop
print("\nTraining...")
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    y_pred = model.forward(X_train, training=True)
    
    # Compute loss
    loss = model.compute_loss(y_train, y_pred)
    
    # Backward pass
    dY = model.loss.backward(y_pred, y_train)
    model.backward(dY)
    
    # Update parameters
    model.step()
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.6f}")

# Test predictions
print("\nFinal Predictions:")
y_pred_final = model.predict_proba(X_train)
for i in range(len(X_train)):
    pred_class = 1 if y_pred_final[i] > 0.5 else 0
    print(f"Input: {X_train[i]} -> Pred: {y_pred_final[i][0]:.4f} -> Class: {pred_class} (True: {int(y_train[i][0])})")


Model Structure:
Model(
  (0) Dense
  (1) Tanh
  (2) Dense
  (3) Sigmoid
)

Training...
Epoch 200/1000 - Loss: 0.075920
Epoch 400/1000 - Loss: 0.057672
Epoch 600/1000 - Loss: 0.054518
Epoch 800/1000 - Loss: 0.053037
Epoch 1000/1000 - Loss: 0.052072

Final Predictions:
Input: [0. 0.] -> Pred: 0.0069 -> Class: 0 (True: 0)
Input: [0. 1.] -> Pred: 0.9898 -> Class: 1 (True: 1)
Input: [1. 0.] -> Pred: 0.9887 -> Class: 1 (True: 1)
Input: [1. 1.] -> Pred: 0.0129 -> Class: 0 (True: 0)


In [4]:
# TEST 2: Regression (Simple Function Approximation)

# Dataset: y = sin(x)
np.random.seed(42)
X_reg = np.linspace(-np.pi, np.pi, 100).reshape(-1, 1)
y_reg = np.sin(X_reg)

# Shuffle
indices = np.random.permutation(len(X_reg))
X_reg = X_reg[indices]
y_reg = y_reg[indices]

# Build model
model_reg = Model(
    modules=[
        Dense(1, 10, initializer=he_uniform),
        ReLU(),
        Dense(10, 10, initializer=he_uniform),
        ReLU(),
        Dense(10, 1, initializer=xavier_uniform),
        Identity()
    ],
    loss=MSE(),
    optimizer=SGD(lr=0.01),
    regularizer=L2(lam=0.0001)
)

print("\nModel Structure:")
print(model_reg)

# Training
print("\nTraining...")
epochs = 500
for epoch in range(epochs):
    y_pred = model_reg.forward(X_reg, training=True)
    loss = model_reg.compute_loss(y_reg, y_pred)
    
    dY = model_reg.loss.backward(y_pred, y_reg)
    model_reg.backward(dY)
    model_reg.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.6f}")

# Test on new points
X_test = np.array([[-np.pi], [0], [np.pi/2], [np.pi]])
y_test_pred = model_reg.predict_proba(X_test)
y_test_true = np.sin(X_test)

print("\nTest Predictions:")
for i in range(len(X_test)):
    print(f"x={X_test[i][0]:6.3f} -> Pred: {y_test_pred[i][0]:6.3f}, True: {y_test_true[i][0]:6.3f}")


Model Structure:
Model(
  (0) Dense
  (1) ReLU
  (2) Dense
  (3) ReLU
  (4) Dense
  (5) Identity
)

Training...
Epoch 100/500 - Loss: 0.197094
Epoch 200/500 - Loss: 0.189761
Epoch 300/500 - Loss: 0.182726
Epoch 400/500 - Loss: 0.175818
Epoch 500/500 - Loss: 0.168948

Test Predictions:
x=-3.142 -> Pred: -0.848, True: -0.000
x= 0.000 -> Pred:  0.001, True:  0.000
x= 1.571 -> Pred:  0.488, True:  1.000
x= 3.142 -> Pred:  0.897, True:  0.000


In [5]:
# TEST 3: Multi-class Classification (Softmax)

# Simple 3-class dataset
np.random.seed(42)
X_multi = np.vstack([
    np.random.randn(20, 2) + np.array([0, 0]),    # Class 0
    np.random.randn(20, 2) + np.array([3, 0]),    # Class 1
    np.random.randn(20, 2) + np.array([1.5, 3])   # Class 2
])

# One-hot encode labels
y_multi = np.zeros((60, 3))
y_multi[0:20, 0] = 1
y_multi[20:40, 1] = 1
y_multi[40:60, 2] = 1

# Shuffle
indices = np.random.permutation(60)
X_multi = X_multi[indices]
y_multi = y_multi[indices]

# Build model
model_multi = Model(
    modules=[
        Dense(2, 8, initializer=he_uniform),
        ReLU(),
        Dense(8, 3, initializer=xavier_uniform),
        Softmax()
    ],
    loss=CrossEntropy(),
    optimizer=SGD(lr=0.1)
)

print("\nModel Structure:")
print(model_multi)

# Training
print("\nTraining...")
epochs = 300
for epoch in range(epochs):
    y_pred = model_multi.forward(X_multi, training=True)
    loss = model_multi.compute_loss(y_multi, y_pred)
    
    dY = model_multi.loss.backward(y_pred, y_multi)
    model_multi.backward(dY)
    model_multi.step()
    
    if (epoch + 1) % 60 == 0:
        # Calculate accuracy
        pred_classes = np.argmax(y_pred, axis=1)
        true_classes = np.argmax(y_multi, axis=1)
        acc = np.mean(pred_classes == true_classes)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f}, Accuracy: {acc:.4f}")


Model Structure:
Model(
  (0) Dense
  (1) ReLU
  (2) Dense
  (3) Softmax
)

Training...
Epoch 60/300 - Loss: 0.2522, Accuracy: 0.9167
Epoch 120/300 - Loss: 0.1974, Accuracy: 0.9167
Epoch 180/300 - Loss: 0.1749, Accuracy: 0.9167
Epoch 240/300 - Loss: 0.1619, Accuracy: 0.9167
Epoch 300/300 - Loss: 0.1530, Accuracy: 0.9333


In [6]:
# TEST 4: Dropout Effect

# Model with dropout
model_drop = Model(
    modules=[
        Dense(2, 8, initializer=he_uniform, seed=42),
        ReLU(),
        Dropout(p=0.5, seed=42),
        Dense(8, 1, initializer=xavier_uniform, seed=42),
        Sigmoid()
    ],
    loss=BinaryCrossEntropy(),
    optimizer=SGD(lr=0.5)
)

print("\nTesting Dropout Behavior...")

# Test input
X_test_drop = np.array([[0.5, 0.5]])

# Training mode (dropout active)
y_pred_train_1 = model_drop.forward(X_test_drop, training=True)
y_pred_train_2 = model_drop.forward(X_test_drop, training=True)

print(f"Training mode (dropout ON):")
print(f"  Pass 1: {y_pred_train_1[0][0]:.6f}")
print(f"  Pass 2: {y_pred_train_2[0][0]:.6f}")
print(f"  Different outputs: {not np.allclose(y_pred_train_1, y_pred_train_2)}")

# Inference mode (dropout off)
y_pred_eval_1 = model_drop.predict_proba(X_test_drop)
y_pred_eval_2 = model_drop.predict_proba(X_test_drop)

print(f"\nInference mode (dropout OFF):")
print(f"  Pass 1: {y_pred_eval_1[0][0]:.6f}")
print(f"  Pass 2: {y_pred_eval_2[0][0]:.6f}")
print(f"  Same outputs: {np.allclose(y_pred_eval_1, y_pred_eval_2)}")


Testing Dropout Behavior...
Training mode (dropout ON):
  Pass 1: 0.500000
  Pass 2: 0.687625
  Different outputs: True

Inference mode (dropout OFF):
  Pass 1: 0.860202
  Pass 2: 0.860202
  Same outputs: True


In [7]:
# TEST 5: Different Losses and Regularizers

X_simple = np.array([[1, 2], [2, 3], [3, 4], [4, 5]], dtype=float)
y_simple = np.array([[3], [5], [7], [9]], dtype=float)

print("\nTesting MSE Loss:")
model_mse = Model(
    modules=[Dense(2, 1), Identity()],
    loss=MSE(),
    optimizer=SGD(lr=0.01)
)
for _ in range(100):
    y_pred = model_mse.forward(X_simple, training=True)
    loss = model_mse.compute_loss(y_simple, y_pred)
    dY = model_mse.loss.backward(y_pred, y_simple)
    model_mse.backward(dY)
    model_mse.step()
print(f"Final MSE Loss: {loss:.6f}")

print("\nTesting MEE Loss:")
model_mee = Model(
    modules=[Dense(2, 1), Identity()],
    loss=MEE(),
    optimizer=SGD(lr=0.01)
)
for _ in range(100):
    y_pred = model_mee.forward(X_simple, training=True)
    loss = model_mee.compute_loss(y_simple, y_pred)
    dY = model_mee.loss.backward(y_pred, y_simple)
    model_mee.backward(dY)
    model_mee.step()
print(f"Final MEE Loss: {loss:.6f}")

print("\nTesting L1 Regularization:")
model_l1 = Model(
    modules=[Dense(2, 4, seed=42), ReLU(), Dense(4, 1, seed=42)],
    loss=MSE(),
    optimizer=SGD(lr=0.01),
    regularizer=L1(lam=0.1)
)
for _ in range(50):
    y_pred = model_l1.forward(X_simple, training=True)
    loss = model_l1.compute_loss(y_simple, y_pred)
    dY = model_l1.loss.backward(y_pred, y_simple)
    model_l1.backward(dY)
    model_l1.step()

# Count near-zero weights (sparsity from L1)
all_weights = []
for m in model_l1.modules:
    if isinstance(m, Dense):
        all_weights.extend(m.W.flatten())
all_weights = np.array(all_weights)
sparse_count = np.sum(np.abs(all_weights) < 0.01)
print(f"L1 Regularization - Weights near zero: {sparse_count}/{len(all_weights)}")


Testing MSE Loss:
Final MSE Loss: 0.071765

Testing MEE Loss:
Final MEE Loss: 0.168436

Testing L1 Regularization:
L1 Regularization - Weights near zero: 1/12


# PROVA DATA LOADER

In [3]:
# --- CONFIGURAZIONE PERCORSI ---
# Modifica questi percorsi in base alla posizione locale dei tuoi file
MONK_TRAIN = "data/MONK/MONK1/monks-1.train"
CUP_TRAIN = "data/CUP/ML-CUP25-TR.csv"

print("=== Inizio Test Validazione Utility ===\n")

# --- 1. TEST MONK DATASET ---
print("1. Test load_monk (MONK1):")
try:
    # Test con One-Hot Encoding (Default)
    X_monk_enc, y_monk = load_monk(MONK_TRAIN, encode=True)
    print(f"   [OK] Monk Encoded - X shape: {X_monk_enc.shape} (Atteso: (N, 17))")
    print(f"   [OK] Monk Encoded - y shape: {y_monk.shape} (Atteso: (N, 1))")
    
    # Verifica che i valori siano solo 0 e 1
    unique_vals = np.unique(X_monk_enc)
    print(f"   [OK] Valori unici in X: {unique_vals} (Atteso: [0. 1.])")

    # Test senza Encoding
    X_monk_raw, _ = load_monk(MONK_TRAIN, encode=False)
    print(f"   [OK] Monk Raw - X shape: {X_monk_raw.shape} (Atteso: (N, 6))")

except Exception as e:
    print(f"   [ERRORE] Test MONK fallito: {e}")

# --- 2. TEST CUP DATASET ---
print("\n2. Test load_cup:")
try:
    X_cup, y_cup = load_cup(CUP_TRAIN, training=True)
    print(f"   [OK] CUP Training - X shape: {X_cup.shape} (Atteso: (N, 12))")
    print(f"   [OK] CUP Training - y shape: {y_cup.shape} (Atteso: (N, 4))")
    
    # Verifica che non ci siano NaN (spesso presenti se il parsing del CSV fallisce)
    if not np.isnan(X_cup).any():
        print("   [OK] Nessun valore mancante (NaN) trovato nei dati CUP.")
    else:
        print("   [WARNING] Trovati valori NaN nei dati CUP!")

except Exception as e:
    print(f"   [ERRORE] Test CUP fallito: {e}")

# --- 3. TEST NORMALIZZAZIONE ---
print("\n3. Test normalize:")
try:
    # Creiamo dati dummy per testare la logica
    dummy_data = np.array([[10, 2], [20, 4], [30, 6]], dtype=float)
    scaled_data, mean, std = normalize(dummy_data)
    
    # Dopo la normalizzazione, la media dovrebbe essere ~0 e la std ~1
    new_mean = np.mean(scaled_data, axis=0)
    new_std = np.std(scaled_data, axis=0)
    
    if np.allclose(new_mean, 0) and np.allclose(new_std, 1):
        print(f"   [OK] Normalizzazione corretta (Media: {new_mean}, Std: {new_std})")
    else:
        print(f"   [ERRORE] La normalizzazione non ha prodotto media 0 e std 1.")

except Exception as e:
    print(f"   [ERRORE] Test Normalizzazione fallito: {e}")

# --- 2.1 TEST CUP TEST SET (Senza target) ---
print("\n2.1 Test load_cup (Test Set):")
CUP_TEST = "data/CUP/ML-CUP25-TS.csv"

try:
    # Caricamento con training=False
    X_cup_test = load_cup(CUP_TEST, training=False)
    
    print(f"   [OK] CUP Test - X shape: {X_cup_test.shape} (Atteso: (N, 12))")
    
    # Verifica che sia stato restituito solo X e non una tupla
    if isinstance(X_cup_test, np.ndarray):
        print("   [OK] Il loader ha restituito correttamente solo la matrice delle feature.")
    else:
        print("   [ERRORE] Il loader ha restituito una tupla, ma il test set non dovrebbe avere target.")

except FileNotFoundError:
    print(f"   [WARNING] File {CUP_TEST} non trovato. Salto questo test.")
except Exception as e:
    print(f"   [ERRORE] Test CUP Test Set fallito: {e}")

print("\n=== Test Completati ===")

=== Inizio Test Validazione Utility ===

1. Test load_monk (MONK1):
   [OK] Monk Encoded - X shape: (124, 17) (Atteso: (N, 17))
   [OK] Monk Encoded - y shape: (124, 1) (Atteso: (N, 1))
   [OK] Valori unici in X: [0. 1.] (Atteso: [0. 1.])
   [OK] Monk Raw - X shape: (124, 6) (Atteso: (N, 6))

2. Test load_cup:
   [OK] CUP Training - X shape: (500, 12) (Atteso: (N, 12))
   [OK] CUP Training - y shape: (500, 4) (Atteso: (N, 4))
   [OK] Nessun valore mancante (NaN) trovato nei dati CUP.

3. Test normalize:
   [OK] Normalizzazione corretta (Media: [0. 0.], Std: [1. 1.])

2.1 Test load_cup (Test Set):
   [OK] CUP Test - X shape: (1000, 12) (Atteso: (N, 12))
   [OK] Il loader ha restituito correttamente solo la matrice delle feature.

=== Test Completati ===


In [4]:
# 1. Carichiamo i dati grezzi
X_cup, y_cup = load_cup("data/CUP/ML-CUP25-TR.csv")

def analyze_ranges(data, title="Analisi"):
    print(f"=== {title} ===")
    # Calcoliamo min, max, media e range (ptp: peak-to-peak) per ogni colonna
    mins = np.min(data, axis=0)
    maxs = np.max(data, axis=0)
    ranges = np.ptp(data, axis=0) # max - min
    means = np.mean(data, axis=0)
    
    print(f"{'Col':>4} | {'Min':>10} | {'Max':>10} | {'Range':>10} | {'Mean':>10}")
    print("-" * 55)
    for i in range(len(mins)):
        print(f"{i:>4} | {mins[i]:>10.2f} | {maxs[i]:>10.2f} | {ranges[i]:>10.2f} | {means[i]:>10.2f}")
    
    # Check critico: calcoliamo il rapporto tra il range più grande e quello più piccolo
    global_max_range = np.max(ranges)
    global_min_range = np.min(ranges)
    ratio = global_max_range / (global_min_range + 1e-8)
    
    print(f"\nRapporto tra range massimo e minimo: {ratio:.2f}")
    if ratio > 10:
        print("--> CONSIGLIO: Le scale sono molto diverse. La normalizzazione è caldamente suggerita.")
    else:
        print("--> CONSIGLIO: Le scale sono simili. Potresti anche evitare la normalizzazione.")

# Analizziamo gli Input (X)
analyze_ranges(X_cup, title="INPUT FEATURES (X)")

print("\n")

# Analizziamo i Target (y)
analyze_ranges(y_cup, title="TARGET VALUES (Y)")

=== INPUT FEATURES (X) ===
 Col |        Min |        Max |      Range |       Mean
-------------------------------------------------------
   0 |     -14.19 |      19.35 |      33.53 |       2.19
   1 |     -13.50 |      18.01 |      31.51 |       1.99
   2 |     -22.97 |      29.44 |      52.40 |       2.96
   3 |     -12.14 |      15.31 |      27.45 |       1.25
   4 |     -11.18 |       8.43 |      19.61 |      -1.16
   5 |      -4.53 |       1.99 |       6.51 |      -1.05
   6 |     -18.19 |      26.35 |      44.54 |       3.34
   7 |     -14.40 |      23.48 |      37.87 |       3.72
   8 |     -13.60 |      18.49 |      32.09 |       1.87
   9 |     -21.86 |      17.08 |      38.94 |      -2.01
  10 |     -12.03 |       9.08 |      21.11 |      -1.10
  11 |     -10.44 |      12.62 |      23.06 |       0.93

Rapporto tra range massimo e minimo: 8.04
--> CONSIGLIO: Le scale sono simili. Potresti anche evitare la normalizzazione.


=== TARGET VALUES (Y) ===
 Col |        Min |      